# Raster least-cost distance

This lab turns a cost surface into a weighted raster graph and runs multi-source Dijkstra. It distinguishes accumulated cost from ordinary Euclidean distance.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt
ROOT = Path.cwd()
if not (ROOT / 'implementations').exists(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'implementations/python/cost-distance'))
from least_cost import least_cost_distance


## 1. Uniform cost surface

Compare four-neighbour and eight-neighbour connectivity. The eight-neighbour graph allows diagonal moves, whose length is calculated from the cell dimensions.

In [ ]:
cost = np.ones((9, 9))
source = [(4, 4)]
d4 = least_cost_distance(cost, source, connectivity=4)
d8 = least_cost_distance(cost, source, connectivity=8)
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, layer, title in zip(axes, [d4, d8], ['4-neighbour', '8-neighbour']):
    im = ax.imshow(layer, cmap='viridis'); ax.scatter([4], [4], marker='*', s=150, c='red'); ax.set_title(title); fig.colorbar(im, ax=ax)
plt.tight_layout(); plt.show()


## 2. High-cost barrier

A high-cost strip encourages the least-cost route to detour. This is a discrete raster path model, so the output is a cost field rather than a continuous path curve.

In [ ]:
cost = np.ones((9, 13)); cost[:, 6] = 25
distance = least_cost_distance(cost, [(4, 1)], connectivity=8)
fig, ax = plt.subplots(figsize=(9, 4))
im = ax.imshow(distance, cmap='magma'); ax.set_title('Accumulated least cost'); fig.colorbar(im, ax=ax, label='Cost units')
plt.tight_layout(); plt.show()


## 3. Barriers and unreachable cells

NoData cells are removed from the traversable graph. Valid cells that cannot be reached from any source remain infinite.

In [ ]:
cost = np.ones((7, 9)); cost[:, 4] = np.nan
distance = least_cost_distance(cost, [(3, 1)])
print('NaN barrier cells:', int(np.isnan(distance).sum()))
print('Unreachable valid cells:', int(np.isinf(distance).sum()))
plt.figure(figsize=(8, 3)); plt.imshow(distance, cmap='viridis'); plt.colorbar(label='Accumulated cost'); plt.title('Barrier separates the raster'); plt.show()


## Investigate

1. Why does 8-neighbour connectivity change accumulated cost?
2. Change the cell dimensions. What happens to diagonal step length?
3. Change the friction values. Does the cheapest route detour?
4. Compare the corner-cutting rule with and without adjacent orthogonal barriers.
5. Why is the result not automatically travel time or a least-cost path geometry?